In [ ]:
# ==============================
# Federated Learning on EMNIST Letters (Dirichlet Non-IID)
# Model: LeNet++ CNN
# Algorithm: FedProx
# Tracks: Train/Val/Test Loss & Accuracy + Round Time + Total Time
# ==============================

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np
import random
import copy
from collections import OrderedDict
import time

# -------------------------
# Configuration
# -------------------------
NUM_CLIENTS = 100
CLIENT_FRAC = 0.3
ROUNDS = 60
LOCAL_EPOCHS = 3
BATCH_SIZE = 128
LR = 0.01
MU = 0.01
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
ALPHA = 0.1

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True

print("Device:", DEVICE)

# -------------------------
# Dataset: EMNIST Letters
# -------------------------
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.EMNIST(
    root="./data",
    split="letters",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.EMNIST(
    root="./data",
    split="letters",
    train=False,
    download=True,
    transform=transform
)

# IMPORTANT: labels are 1–26 → convert to 0–25
train_dataset.targets = train_dataset.targets - 1
test_dataset.targets = test_dataset.targets - 1

NUM_CLASSES = 26

# -------------------------
# Validation Split
# -------------------------
VAL_CLIENTS = list(range(10))
TRAIN_CLIENTS = list(range(10, NUM_CLIENTS))

# -------------------------
# Dirichlet Non-IID Partition
# -------------------------
def dirichlet_partition(dataset, clients, alpha=0.1):
    labels = np.array(dataset.targets)
    idxs = np.arange(len(labels))
    client_dict = {i: [] for i in clients}

    class_indices = [idxs[labels == i] for i in range(NUM_CLASSES)]

    for c_idx in class_indices:
        np.random.shuffle(c_idx)
        proportions = np.random.dirichlet(alpha * np.ones(len(clients)))
        proportions = (np.cumsum(proportions) * len(c_idx)).astype(int)
        proportions = np.concatenate(([0], proportions))

        for i, client in enumerate(clients):
            client_dict[client] += list(c_idx[proportions[i]:proportions[i+1]])

    for client in clients:
        np.random.shuffle(client_dict[client])

    return client_dict

client_data = dirichlet_partition(train_dataset, list(range(NUM_CLIENTS)), alpha=ALPHA)

# -------------------------
# Model: CNN (LeNet++)
# -------------------------
class EMNIST_CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc1 = nn.Linear(64 * 7 * 7, 256)
        self.fc2 = nn.Linear(256, NUM_CLASSES)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

# -------------------------
# FedProx Regularizer
# -------------------------
def fedprox_loss(model, global_model, mu):
    prox = 0.0
    for w, w0 in zip(model.parameters(), global_model.parameters()):
        prox += torch.sum((w - w0.detach()) ** 2)
    return mu / 2 * prox

# -------------------------
# Evaluation
# -------------------------
def evaluate(model, loader):
    model.eval()
    correct, total, loss_val = 0, 0, 0.0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss_val += F.cross_entropy(out, y).item() * y.size(0)
            pred = out.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)

    return 100 * correct / total, loss_val / total

# -------------------------
# Federated Training
# -------------------------
global_model = EMNIST_CNN().to(DEVICE)

train_acc, val_acc, test_acc = [], [], []
loss_curve, train_loss_curve, val_loss_curve, test_loss_curve = [], [], [], []
round_time, total_time = [], []

start_total = time.time()

for rnd in range(ROUNDS):
    start_round = time.time()

    local_weights = []
    local_sizes = []
    local_losses = []

    selected_clients = random.sample(TRAIN_CLIENTS, int(NUM_CLIENTS * CLIENT_FRAC))

    for client in selected_clients:
        local_model = copy.deepcopy(global_model)
        local_model.train()

        optimizer = optim.SGD(local_model.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)

        loader = DataLoader(
            Subset(train_dataset, client_data[client]),
            batch_size=BATCH_SIZE,
            shuffle=True
        )

        for _ in range(LOCAL_EPOCHS):
            for x, y in loader:
                x, y = x.to(DEVICE), y.to(DEVICE)

                optimizer.zero_grad()
                out = local_model(x)

                loss = F.cross_entropy(out, y)
                loss += fedprox_loss(local_model, global_model, MU)

                loss.backward()
                optimizer.step()

        local_losses.append(loss.item())
        local_weights.append(copy.deepcopy(local_model.state_dict()))
        local_sizes.append(len(client_data[client]))

    # -------------------------
    # Aggregation
    # -------------------------
    total = sum(local_sizes)
    new_weights = OrderedDict()

    for k in global_model.state_dict().keys():
        new_weights[k] = sum(
            local_weights[i][k] * (local_sizes[i] / total)
            for i in range(len(local_weights))
        )

    global_model.load_state_dict(new_weights)

    # -------------------------
    # Metrics
    # -------------------------
    train_loader = DataLoader(
        torch.utils.data.ConcatDataset([
            Subset(train_dataset, client_data[c]) for c in selected_clients
        ]),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    val_loader = DataLoader(
        torch.utils.data.ConcatDataset([
            Subset(train_dataset, client_data[c]) for c in VAL_CLIENTS
        ]),
        batch_size=BATCH_SIZE,
        shuffle=False
    )

    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

    tr_acc, tr_loss = evaluate(global_model, train_loader)
    va_acc, va_loss = evaluate(global_model, val_loader)
    te_acc, te_loss = evaluate(global_model, test_loader)

    loss_curve.append(np.mean(local_losses))
    train_loss_curve.append(tr_loss)
    val_loss_curve.append(va_loss)
    test_loss_curve.append(te_loss)

    train_acc.append(tr_acc)
    val_acc.append(va_acc)
    test_acc.append(te_acc)

    rt = time.time() - start_round
    round_time.append(rt)
    total_time.append(time.time() - start_total)

    print(f"Round {rnd+1:03d} | "
          f"LocalLoss {loss_curve[-1]:.4f} | "
          f"TrainLoss {tr_loss:.4f} | "
          f"ValLoss {va_loss:.4f} | "
          f"TestLoss {te_loss:.4f} | "
          f"Train {tr_acc:.4f} | Val {va_acc:.4f} | Test {te_acc:.4f} | "
          f"RoundTime {rt:.2f}s | TotalTime {total_time[-1]:.2f}s")

In [ ]:
# -------------------------
# Save metrics to CSV
# -------------------------
import csv

metrics = {
    "round": list(range(1, ROUNDS+1)),
    "train_acc": train_acc,
    "val_acc": val_acc,
    "test_acc": test_acc,
    "local_loss": loss_curve,
    "train_loss": train_loss_curve,
    "val_loss": val_loss_curve,
    "test_loss": test_loss_curve,
    "round_time": round_time,
    "total_time": total_time
}

with open("EMNIST_Letters_fedprox_federated_metrics_0.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(metrics.keys())
    writer.writerows(zip(*metrics.values()))

# Colab download
try:
    from google.colab import files
    files.download("EMNIST_Letters_fedprox_federated_metrics_0.csv")
except:
    print("CSV saved locally as EMNIST_Letters_fedprox_federated_metrics_0.csv")

In [ ]:
# -------------------------
# Save Global Model
# -------------------------
MODEL_PATH = "Baseline_emnist_letters.pth"

torch.save(global_model.state_dict(), MODEL_PATH)

print("Model saved to:", MODEL_PATH)

# -------------------------
# Download model to laptop
# -------------------------
from google.colab import files
files.download(MODEL_PATH)